# VCD Table 1 Reproduction
Runs LLaVA-1.5 on POPE `random` and `popular` with `regular` vs `vcd`, then reports Accuracy/Precision/Recall/F1.

In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU not enabled.')

Torch: 2.9.0+cu128
CUDA available: True
GPU: NVIDIA H100 80GB HBM3


In [ ]:
# Config
REPO_URL = 'https://github.com/maxheef/ECE209_Project1.git'
BRANCH = 'main'
# Parameters from original file: 'llava1.5_pope.bash'
MODEL_PATH = 'liuhaotian/llava-v1.5-7b' 
SEED = 55
CD_ALPHA = 1.0
CD_BETA = 0.2
NOISE_STEP = 500
COCO_VAL_DIR = '/content/datasets/coco/val2014'

# Create temporary files inside of Google Colab
from pathlib import Path
env_file = Path('/tmp/vcd_params.env')
env_file.write_text(
    f"REPO_URL='{REPO_URL}'\n"
    f"BRANCH='{BRANCH}'\n"
    f"MODEL_PATH='{MODEL_PATH}'\n"
    f"SEED='{SEED}'\n"
    f"CD_ALPHA='{CD_ALPHA}'\n"
    f"CD_BETA='{CD_BETA}'\n"
    f"NOISE_STEP='{NOISE_STEP}'\n"
    f"COCO_VAL_DIR='{COCO_VAL_DIR}'\n"
)
print(env_file.read_text())

REPO_URL='https://github.com/maxheef/ECE209_Project1.git'
BRANCH='main'
MODEL_PATH='liuhaotian/llava-v1.5-7b'
SEED='55'
CD_ALPHA='1.0'
CD_BETA='0.2'
NOISE_STEP='500'
COCO_VAL_DIR='/content/datasets/coco/val2014'



In [ ]:
%%bash
set -euo pipefail
source /tmp/vcd_params.env

# Check if using the most up to date project from Git
cd /content
if [ -d /content/VCD_project/.git ]; then
  git -C /content/VCD_project fetch --depth 1 origin "$BRANCH"
  git -C /content/VCD_project reset --hard FETCH_HEAD
else
  rm -rf /content/VCD_project
  GIT_TERMINAL_PROMPT=0 git clone --depth 1 --branch "$BRANCH" "$REPO_URL" /content/VCD_project
fi

# Confirm the file structure
if [ -d /content/VCD_project/VCD ] && [ -d /content/VCD_project/myCode ]; then
  PROJ=/content/VCD_project
elif [ -d /content/VCD_project/VCD_project/VCD ] && [ -d /content/VCD_project/VCD_project/myCode ]; then
  PROJ=/content/VCD_project/VCD_project
else
  echo 'File structure should be: /content/VCD_project'
  find /content/VCD_project -maxdepth 3 -type d | sed -n '1,120p'
  exit 1
fi
echo "$PROJ" > /tmp/vcd_project_root.txt
echo "Using project root: $PROJ"
ls -la "$PROJ"

Using project root: /content/VCD_project
total 116
drwxr-xr-x 6 root root  4096 Feb 17 01:46 .
drwxr-xr-x 1 root root  4096 Feb 17 01:46 ..
-rw-r--r-- 1 root root 86319 Feb 17 01:46 2026W Trustworthy AI Individual Project Writeup.pdf
drwxr-xr-x 8 root root  4096 Feb 17 01:46 .git
-rw-r--r-- 1 root root   334 Feb 17 01:46 .gitignore
drwxr-xr-x 2 root root  4096 Feb 17 01:46 myCode
drwxr-xr-x 2 root root  4096 Feb 17 01:46 Papers
drwxr-xr-x 5 root root  4096 Feb 17 01:46 VCD


Cloning into '/content/VCD_project'...


In [ ]:
%%bash
set -euo pipefail
PROJ=$(cat /tmp/vcd_project_root.txt)

if [ ! -x /usr/local/miniconda/bin/conda ]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
  bash /tmp/miniconda.sh -b -p /usr/local/miniconda
fi
source /usr/local/miniconda/etc/profile.d/conda.sh
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main || true
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r || true
conda config --set always_yes yes --set changeps1 no
if ! conda env list | grep -q '^vcd39[[:space:]]'; then
  conda create -n vcd39 python=3.9
fi
# Need to use Python 3.9 from the requirements to be compatible
conda activate vcd39
python -m pip install --upgrade pip
cd "$PROJ/VCD"
pip install -r requirements.txt
# Setup PyTorch for H100 compatibility
pip install --upgrade --index-url https://download.pytorch.org/whl/cu121 torch==2.2.2 torchvision==0.17.2
pip install --force-reinstall 'numpy<2'
python - <<'PYCHK'
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
PYCHK
echo /usr/local/miniconda/envs/vcd39/bin/python > /tmp/vcd_python_bin.txt
python -V

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.3/757.3 MB 15.5 MB/s  0:00:44
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 6.3 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 238.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 57.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 237.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 57.5 MB/s  0:00:06
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 60.0 MB/s  0:00:05
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 151.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 170.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 142.5 MB/s  0:00:00
     ━━━━━━━━━━━━━

In [ ]:
%%bash
set -euo pipefail
# Download the Val2014 image dataset
source /tmp/vcd_params.env
mkdir -p /content/datasets/coco
cd /content/datasets/coco
if [ ! -d val2014 ]; then
  wget -q http://images.cocodataset.org/zips/val2014.zip
  unzip -q val2014.zip
fi
ls -la "$COCO_VAL_DIR" | sed -n '1,10p'

total 6564388
drwxrwxr-x 2 root root 2351104 Aug 16  2014 .
drwxr-xr-x 3 root root    4096 Feb 17 01:57 ..
-rw-rw-r-- 1 root root  213308 Aug 16  2014 COCO_val2014_000000000042.jpg
-rw-rw-r-- 1 root root  383651 Aug 16  2014 COCO_val2014_000000000073.jpg
-rw-rw-r-- 1 root root  176151 Aug 16  2014 COCO_val2014_000000000074.jpg
-rw-rw-r-- 1 root root  163607 Aug 16  2014 COCO_val2014_000000000133.jpg
-rw-rw-r-- 1 root root  105082 Aug 16  2014 COCO_val2014_000000000136.jpg
-rw-rw-r-- 1 root root  161811 Aug 16  2014 COCO_val2014_000000000139.jpg
-rw-rw-r-- 1 root root   59554 Aug 16  2014 COCO_val2014_000000000143.jpg


In [ ]:
%%bash
set -euo pipefail
# Utilize the correct Parameters
source /tmp/vcd_params.env
PROJ=$(cat /tmp/vcd_project_root.txt)
PYBIN=$(cat /tmp/vcd_python_bin.txt)

cd "$PROJ"
PYTHON_BIN="$PYBIN" \
CD_ALPHA="$CD_ALPHA" CD_BETA="$CD_BETA" NOISE_STEP="$NOISE_STEP" \
# Run the Model
bash myCode/run_table1_repro.sh \
  "$MODEL_PATH" \
  "$COCO_VAL_DIR" \
  "$SEED"

Running split=random method=regular
Saved: /content/VCD_project/myCode/output/llava15_coco_pope_random_answers_regular_seed55.jsonl
Saved: /content/VCD_project/myCode/output/metrics_random_regular_seed55.json
Running split=random method=vcd
Saved: /content/VCD_project/myCode/output/llava15_coco_pope_random_answers_vcd_seed55.jsonl
Saved: /content/VCD_project/myCode/output/metrics_random_vcd_seed55.json
Running split=popular method=regular
Saved: /content/VCD_project/myCode/output/llava15_coco_pope_popular_answers_regular_seed55.jsonl
Saved: /content/VCD_project/myCode/output/metrics_popular_regular_seed55.json
Running split=popular method=vcd
Saved: /content/VCD_project/myCode/output/llava15_coco_pope_popular_answers_vcd_seed55.jsonl
Saved: /content/VCD_project/myCode/output/metrics_popular_vcd_seed55.json
| Split | Method | Accuracy | Precision | Recall | F1 |
|---|---|---:|---:|---:|---:|
| random | regular | 0.8293 | 0.9201 | 0.7213 | 0.8087 |
| random | vcd | 0.8553 | 0.9369 | 0.76

/usr/local/miniconda/envs/vcd39/lib/python3.9/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
100%|██████████| 3000/3000 [03:46<00:00, 13.25it/s]
/usr/local/miniconda/envs/vcd39/lib/python3.9/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
100%|██████████| 3000/3000 [06:37<00:00,  7.55it/s]
/usr/local/miniconda/envs/vcd39/lib/python3.9/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  wa

In [11]:
%%bash
set -euo pipefail
source /tmp/vcd_params.env
PROJ=$(cat /tmp/vcd_project_root.txt)
cd "$PROJ"
ls -la myCode/output
echo '\n--- random regular ---'
cat myCode/output/metrics_random_regular_seed${SEED}.json
echo '\n--- random vcd ---'
cat myCode/output/metrics_random_vcd_seed${SEED}.json
echo '\n--- popular regular ---'
cat myCode/output/metrics_popular_regular_seed${SEED}.json
echo '\n--- popular vcd ---'
cat myCode/output/metrics_popular_vcd_seed${SEED}.json
echo '\n--- table ---'
python3 myCode/table1_from_outputs.py --outputs myCode/output --gt-root VCD/experiments/data/POPE/coco --seed "$SEED"

total 2024
drwxr-xr-x 2 root root   4096 Feb 17 02:27 .
drwxr-xr-x 4 root root   4096 Feb 17 02:27 ..
-rw-r--r-- 1 root root 504920 Feb 17 02:20 llava15_coco_pope_popular_answers_regular_seed55.jsonl
-rw-r--r-- 1 root root 504968 Feb 17 02:27 llava15_coco_pope_popular_answers_vcd_seed55.jsonl
-rw-r--r-- 1 root root 505842 Feb 17 02:09 llava15_coco_pope_random_answers_regular_seed55.jsonl
-rw-r--r-- 1 root root 505886 Feb 17 02:16 llava15_coco_pope_random_answers_vcd_seed55.jsonl
-rw-r--r-- 1 root root    265 Feb 17 02:20 metrics_popular_regular_seed55.json
-rw-r--r-- 1 root root    240 Feb 17 02:27 metrics_popular_vcd_seed55.json
-rw-r--r-- 1 root root    265 Feb 17 02:09 metrics_random_regular_seed55.json
-rw-r--r-- 1 root root    265 Feb 17 02:16 metrics_random_vcd_seed55.json
\n--- random regular ---
{
  "accuracy": 0.8293333333333334,
  "f1": 0.8086696562032885,
  "false_neg": 418,
  "false_pos": 94,
  "precision": 0.9200680272108843,
  "recall": 0.7213333333333334,
  "total": 3000